In [6]:
#!/usr/bin/env python3
"""
FakeShield‑T: Lightweight Cross‑Attention Transformer for Image Forgery Detection
Full Thesis Implementation — Kaggle Notebook
===================================================================================
Implements all research sections from the proposal using:
  - DF2023 (splicing/inpainting)
  - DEFACTO (copy-move)
  - CIFAKE (AI-generated vs real)
  - TrueFake (platform‑laundered GAN/Diffusion vs real)

Architecture:
  Stream 1 : SRM‑EfficientNet‑B0  (local artifact detection)
  Stream 2 : TinyViT‑5M           (global semantic reasoning)
  Fusion   : Cross‑Attention      (metadata‑aware, missing‑modality robust)
  Head     : DS‑FPN + PANet       (pixel‑level localization)
  XAI      : Grad‑CAM++ + Attention rollout
"""

# ============================================================
# SECTION 0 ─ Environment & Imports
# ============================================================
import os, sys, json, time, random, warnings, math
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import functional as TF
import torchvision.transforms.functional as TVF
from PIL import Image, ImageFilter
import cv2
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, classification_report, roc_curve
)

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[ENV] Device: {DEVICE}")
print(f"[ENV] PyTorch: {torch.__version__}")
print(f"[ENV] CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[ENV] GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# SECTION 1 ─ Dataset Paths & Configuration
# ============================================================
ROOT = Path('/kaggle/input/datasets')

# ── DF2023 (splicing / inpainting) ──────────────────────────
DF2023_IMG  = ROOT / 'hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15'
DF2023_MASK = ROOT / 'hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15_GT'

# ── DEFACTO copy‑move ────────────────────────────────────────
DEFACTO_IMG   = ROOT / 'defactodataset/defactocopymove/copymove_img/img'
DEFACTO_DONOR = ROOT / 'defactodataset/defactocopymove/copymove_annotations/donor_mask'
DEFACTO_PROBE = ROOT / 'defactodataset/defactocopymove/copymove_annotations/probe_mask'

# ── CIFAKE ───────────────────────────────────────────────────
CIFAKE_TRAIN = ROOT / 'birdy654/cifake-real-and-ai-generated-synthetic-images/train'
CIFAKE_TEST  = ROOT / 'birdy654/cifake-real-and-ai-generated-synthetic-images/test'

# ── TrueFake (fixed path) ────────────────────────────────────
# TrueFake dataset is stored under: mubarekahmed/trurfake/truefake_full/truefake_full/
# Inside: Facebook/{Fake,Real}, Telegram/{Fake,Real}
# Fake contains subdirs per model (e.g. FLUX.1/animal, faces, ...)
# Real contains subdirs FFHQ & FORLAB (FORLAB may use .gpg extension)
TRUEFAKE = ROOT / 'mubarekahmed/trurfake/truefake_full/truefake_full'

# ── Hyperparameters ──────────────────────────────────────────
CFG = dict(
    img_size        = 256,
    batch_size      = 32,
    num_workers     = 4,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    epochs          = 30,
    warmup_epochs   = 3,
    label_smoothing = 0.1,
    dropout         = 0.3,
    embed_dim       = 256,
    num_heads       = 8,
    max_samples_per_src = 5000,
    val_split       = 0.15,
    test_split      = 0.10,
    metadata_drop_p = 0.5,
    platforms       = ['Telegram', 'Facebook', 'WhatsApp', 'Twitter'],
)
print(f"[CFG] Image size: {CFG['img_size']}x{CFG['img_size']}")
print(f"[CFG] Batch size: {CFG['batch_size']}, Epochs: {CFG['epochs']}")

# ============================================================
# SECTION 2 ─ Data Preprocessing & Platform Distortion
# ============================================================

class PlatformDistortion:
    """Simulates OSN processing pipelines (Section 3.3.2)."""
    CONFIGS = {
        'Telegram':  dict(stages=2, q_range=(65, 85), resize=True,  chroma=False),
        'WhatsApp':  dict(stages=1, q_range=(40, 60), resize=True,  chroma=False),
        'Facebook':  dict(stages=2, q_range=(70, 90), resize=True,  chroma=True),
        'Twitter':   dict(stages=1, q_range=(55, 75), resize=False, chroma=True),
        'clean':     dict(stages=0, q_range=(95,100), resize=False, chroma=False),
    }

    @staticmethod
    def apply(img: Image.Image, platform: str = 'clean') -> Image.Image:
        cfg = PlatformDistortion.CONFIGS.get(platform, PlatformDistortion.CONFIGS['clean'])
        import io
        if cfg['resize'] and random.random() < 0.5:
            scale = random.uniform(0.7, 0.95)
            w, h = img.size
            img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
            img = img.resize((w, h), Image.LANCZOS)
        for _ in range(cfg['stages']):
            q = random.randint(*cfg['q_range'])
            buf = io.BytesIO()
            subsampling = 2 if cfg['chroma'] else 0
            img.save(buf, format='JPEG', quality=q, subsampling=subsampling)
            buf.seek(0)
            img = Image.open(buf).copy()
        return img

    @staticmethod
    def random_platform() -> str:
        return random.choice(list(PlatformDistortion.CONFIGS.keys()))


class SRMFilters(nn.Module):
    """30 hand‑crafted SRM high‑pass filters (Section 3.4.1)."""
    def __init__(self):
        super().__init__()
        kernels = self._build_srm_kernels()
        weight = torch.zeros(30, 3, 5, 5)
        for i, k in enumerate(kernels[:30]):
            k_t = torch.tensor(k, dtype=torch.float32)
            for c in range(3):
                weight[i, c] = k_t
        self.register_buffer('weight', weight)
        self.register_buffer('bias', torch.zeros(30))

    @staticmethod
    def _build_srm_kernels():
        srm1 = np.array([[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]]) / 4.0
        srm2 = np.array([[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]]) / 12.0
        srm3 = np.array([[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]]) / 2.0
        srm4 = np.array([[0,0,0,0,0],[0,0,1,0,0],[0,1,-4,1,0],[0,0,1,0,0],[0,0,0,0,0]])
        srm5 = np.array([[-1,0,0,0,0],[0,-1,0,0,0],[0,0,4,0,0],[0,0,0,-1,0],[0,0,0,0,-1]])
        base = [srm1, srm2, srm3, srm4, srm5]
        kernels = []
        for k in base:
            kernels.extend([k, -k, np.rot90(k), np.rot90(k, 2)])
        kernels.extend([srm1, srm2, srm3, srm4, srm5, srm1+srm3])
        return kernels[:30]

    def forward(self, x):
        return F.conv2d(x, self.weight, self.bias, padding=2)


# ============================================================
# SECTION 3 ─ Dataset Builders (with corrected TrueFake traversal)
# ============================================================

def collect_truefake_samples(root: Path, max_per_src: int = 2000) -> List[dict]:
    """
    Traverse TrueFake directory tree:
        Facebook/{Fake/{model}/{category}, Real/FFHQ, Real/FORLAB}
        Telegram/{Fake/{model}/{category}, Real/FFHQ, Real/FORLAB}
    Fake categories: animal, landscape animals, faces, general
    Real: FFHQ (jpg), FORLAB (may be .gpg)
    """
    samples = []
    AI_MODELS = ['FLUX.1','StableDiffusion1.5','StableDiffusion2',
                 'StableDiffusion3','StableDiffusionXL',
                 'StyleGAN','StyleGAN2','StyleGAN3']
    for platform in ['Facebook', 'Telegram']:
        plat_path = root / platform
        if not plat_path.exists():
            print(f"  [WARN] TrueFake/{platform} not found, skipping.")
            continue

        # ── Fake ─────────────────────────────────────────────────
        fake_root = plat_path / 'Fake'
        if fake_root.exists():
            for model in AI_MODELS:
                model_dir = fake_root / model
                if not model_dir.is_dir():
                    continue
                # model_dir contains subdirectories like animal, landscape animals, faces, general
                category_dirs = [d for d in model_dir.iterdir() if d.is_dir()]
                for cat_dir in category_dirs:
                    imgs = list(cat_dir.glob('*.jpg')) + list(cat_dir.glob('*.png'))
                    random.shuffle(imgs)
                    for p in imgs[:max_per_src]:
                        samples.append(dict(
                            img_path=str(p), mask_path=None, label=1,
                            manip_type='ai_generated',
                            source=f'TrueFake_{platform}_{model}',
                            platform=platform
                        ))

        # ── Real ─────────────────────────────────────────────────
        real_root = plat_path / 'Real'
        if real_root.exists():
            for src_name in ['FFHQ', 'FORLAB']:
                src_dir = real_root / src_name
                if not src_dir.is_dir():
                    continue
                imgs = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.png'))
                # FORLAB may contain .gpg files that can be opened as images
                if src_name == 'FORLAB':
                    imgs += list(src_dir.glob('*.gpg'))
                random.shuffle(imgs)
                for p in imgs[:max_per_src]:
                    samples.append(dict(
                        img_path=str(p), mask_path=None, label=0,
                        manip_type='real',
                        source=f'TrueFake_{platform}_{src_name}',
                        platform=platform
                    ))
    return samples


def collect_df2023_samples(img_dir: Path, mask_dir: Path, max_n: int = 5000) -> List[dict]:
    samples = []
    if not img_dir.exists():
        print(f"  [WARN] DF2023 img dir not found: {img_dir}")
        return samples
    imgs = sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png'))
    random.shuffle(imgs)
    for p in imgs[:max_n]:
        stem = p.stem
        mask = mask_dir / (stem + '.png')
        if not mask.exists():
            mask = mask_dir / (stem + '.jpg')
        samples.append(dict(
            img_path=str(p),
            mask_path=str(mask) if mask.exists() else None,
            label=1, manip_type='splicing',
            source='DF2023', platform='clean'
        ))
    return samples


def collect_defacto_samples(img_dir: Path, probe_dir: Path, max_n: int = 5000) -> List[dict]:
    samples = []
    if not img_dir.exists():
        print(f"  [WARN] DEFACTO img dir not found: {img_dir}")
        return samples
    imgs = sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.tif'))
    random.shuffle(imgs)
    for p in imgs[:max_n]:
        stem = p.stem
        mask = probe_dir / (stem + '.png')
        samples.append(dict(
            img_path=str(p),
            mask_path=str(mask) if mask.exists() else None,
            label=1, manip_type='copy_move',
            source='DEFACTO', platform='clean'
        ))
    return samples


def collect_cifake_samples(root: Path, max_n: int = 5000) -> List[dict]:
    samples = []
    if not root.exists():
        print(f"  [WARN] CIFAKE dir not found: {root}")
        return samples
    for label_name, label in [('FAKE', 1), ('REAL', 0)]:
        d = root / label_name
        if not d.is_dir():
            continue
        imgs = list(d.glob('*.jpg')) + list(d.glob('*.png'))
        random.shuffle(imgs)
        for p in imgs[:max_n]:
            samples.append(dict(
                img_path=str(p), mask_path=None,
                label=label,
                manip_type='ai_generated' if label else 'real',
                source='CIFAKE', platform='clean'
            ))
    return samples


def build_master_dataframe() -> pd.DataFrame:
    print("\n[DATA] Building master dataset index...")
    all_samples = []

    tf_samples = collect_truefake_samples(TRUEFAKE, max_per_src=CFG['max_samples_per_src']//8)
    print(f"  TrueFake: {len(tf_samples)} samples")
    all_samples.extend(tf_samples)

    df23 = collect_df2023_samples(DF2023_IMG, DF2023_MASK, max_n=CFG['max_samples_per_src'])
    print(f"  DF2023:   {len(df23)} samples")
    all_samples.extend(df23)

    defacto = collect_defacto_samples(DEFACTO_IMG, DEFACTO_PROBE, max_n=CFG['max_samples_per_src'])
    print(f"  DEFACTO:  {len(defacto)} samples")
    all_samples.extend(defacto)

    cifake = collect_cifake_samples(CIFAKE_TRAIN, max_n=CFG['max_samples_per_src'])
    print(f"  CIFAKE:   {len(cifake)} samples")
    all_samples.extend(cifake)

    df = pd.DataFrame(all_samples)
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    n = len(df)
    n_test = int(n * CFG['test_split'])
    n_val  = int(n * CFG['val_split'])
    df['split'] = 'train'
    df.iloc[:n_test, df.columns.get_loc('split')] = 'test'
    df.iloc[n_test:n_test+n_val, df.columns.get_loc('split')] = 'val'

    print(f"\n  Total: {len(df)} samples")
    print(f"  Train: {(df.split=='train').sum()}, Val: {(df.split=='val').sum()}, Test: {(df.split=='test').sum()}")
    print(f"  Real:  {(df.label==0).sum()}, Fake: {(df.label==1).sum()}")
    return df


# ============================================================
# SECTION 4 ─ PyTorch Dataset (unchanged)
# ============================================================

class FakeShieldDataset(Dataset):
    IMG_MEAN = [0.485, 0.456, 0.406]
    IMG_STD  = [0.229, 0.224, 0.225]

    def __init__(self, df: pd.DataFrame, img_size: int = 256,
                 augment: bool = True, metadata_drop_p: float = 0.5):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.augment  = augment
        self.meta_drop_p = metadata_drop_p
        self.normalize = transforms.Normalize(self.IMG_MEAN, self.IMG_STD)
        self.to_tensor = transforms.ToTensor()

    def __len__(self): return len(self.df)

    def _load_img(self, path: str) -> Image.Image:
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            return Image.new('RGB', (self.img_size, self.img_size), (128,128,128))

    def _load_mask(self, path: Optional[str]) -> torch.Tensor:
        if path is None: return torch.zeros(1, self.img_size, self.img_size)
        try:
            mask = Image.open(path).convert('L').resize((self.img_size, self.img_size), Image.NEAREST)
            return (torch.from_numpy(np.array(mask)).float()/255.0 > 0.5).float().unsqueeze(0)
        except:
            return torch.zeros(1, self.img_size, self.img_size)

    def _pad_to_square(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        if w == h: return img
        side = max(w, h)
        new = Image.new('RGB', (side, side), (0,0,0))
        new.paste(img, ((side-w)//2, (side-h)//2))
        return new

    def _augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.3: img = img.transpose(Image.FLIP_TOP_BOTTOM)
        img = img.rotate(random.uniform(-15,15), fillcolor=(128,128,128))
        img = transforms.ColorJitter(0.2,0.2,0.2,0.05)(img)
        plat = PlatformDistortion.random_platform()
        return PlatformDistortion.apply(img, plat)

    def _extract_metadata(self, img: Image.Image) -> torch.Tensor:
        feats = np.zeros(16, dtype=np.float32)
        arr = np.array(img).astype(np.float32)/255.0
        feats[:3] = arr.mean(axis=(0,1))
        feats[3] = arr.std()
        feats[4] = img.width/1920.0
        feats[5] = img.height/1080.0
        gray = arr.mean(axis=2)
        lap = cv2.Laplacian((gray*255).astype(np.uint8), cv2.CV_64F).var()
        feats[6] = float(lap)/1e6
        feats[7] = float(np.corrcoef(arr[:,:,0].ravel(), arr[:,:,1].ravel())[0,1])
        feats[8] = float(np.corrcoef(arr[:,:,1].ravel(), arr[:,:,2].ravel())[0,1])
        feats[9] = float(np.corrcoef(arr[:,:,0].ravel(), arr[:,:,2].ravel())[0,1])
        block = gray[:32,:32]
        dct = cv2.dct(block.astype(np.float32))
        feats[10] = float(np.abs(dct[4:,4:]).mean())
        blurred = cv2.GaussianBlur(gray, (5,5),0)
        noise = gray - blurred
        feats[11] = noise.mean(); feats[12] = noise.std()
        hist,_ = np.histogram(gray, bins=32, density=True); hist += 1e-8
        feats[13] = float(-np.sum(hist*np.log(hist)))
        feats[14] = arr.max(); feats[15] = arr.min()
        return torch.tensor(feats, dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_img(str(row['img_path']))
        img = self._pad_to_square(img)
        if self.augment:
            img = self._augment(img)
        img_rs = img.resize((self.img_size, self.img_size), Image.LANCZOS)
        meta = self._extract_metadata(img_rs)
        mask = self._load_mask(row.get('mask_path'))
        label = int(row['label'])
        if self.augment and random.random() < self.meta_drop_p:
            meta = torch.zeros_like(meta)
        img_t = self.normalize(self.to_tensor(img_rs))
        return dict(image=img_t, metadata=meta, label=torch.tensor(label, dtype=torch.long),
                    mask=mask, source=str(row.get('source','')), manip_type=str(row.get('manip_type','')))


# ============================================================
# (Rest of the model, training, evaluation code remains identical)
# ... [From SECTION 5 onward, unchanged except for ensuring all functions are present]
# ============================================================

# [INCLUDE HERE THE FULL SECTION 5 (Model Architecture), SECTION 6 (Baselines),
#  SECTION 7 (Training Infrastructure), SECTION 8 (Training & Evaluation Loops),
#  SECTION 9 (Platform Robustness), SECTION 10 (XAI), SECTION 11 (Visualisation),
#  and SECTION 12 (Main Pipeline) from the previous script.]
# The complete script with these sections should be used.
# For brevity, only the modified dataset part is shown above; the rest is identical
# to the original code but with the corrected TrueFake path and traversal.

[ENV] Device: cpu
[ENV] PyTorch: 2.10.0+cpu
[ENV] CUDA available: False
[CFG] Image size: 256x256
[CFG] Batch size: 32, Epochs: 30
